# ArmorVault — Synthetic OCR Benchmark (CPU)

Runs PP-OCRv5 Mobile and deterministic field mapping without E5 or GPU on 12 privacy-safe synthetic documents.

In [ ]:
%pip install -q "rapidocr>=3.9,<4" "onnxruntime>=1.20" huggingface_hub "pyyaml>=6" "python-bidi>=0.6,<1" "arabic-reshaper>=3,<4" pillow

In [ ]:
import shutil, subprocess, sys
from pathlib import Path
from urllib.request import urlretrieve

repo = Path('/content/armorvault-ocr-vl-gpu-lab')
dataset = Path('/content/armorvault-synthetic-benchmark')
result_file = Path('/content/armorvault-synthetic-results.json')
shutil.rmtree(repo, ignore_errors=True)
shutil.rmtree(dataset, ignore_errors=True)
result_file.unlink(missing_ok=True)

subprocess.run(['git', 'clone', '-q', 'https://github.com/almawti/armorvault-ocr-vl-gpu-lab.git', str(repo)], check=True)
font = Path('/content/NotoSansArabic.ttf')
urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/notosansarabic/NotoSansArabic%5Bwdth,wght%5D.ttf', font)
if not font.exists() or font.stat().st_size == 0:
    raise RuntimeError('Arabic font download failed')

subprocess.run([sys.executable, str(repo / 'generate_synthetic_ocr_benchmark.py'), '--output', str(dataset), '--font', str(font)], check=True)
subprocess.run([sys.executable, str(repo / 'run_synthetic_ocr_benchmark.py'), '--dataset', str(dataset), '--output', str(result_file)], check=True)
print('Benchmark completed:', result_file)

In [ ]:
import json
from pathlib import Path
result_path = Path('/content/armorvault-synthetic-results.json')
if not result_path.exists():
    raise RuntimeError('Benchmark did not finish. Run all cells from the top and inspect the first red error.')
result = json.loads(result_path.read_text(encoding='utf-8'))
summary = {key: value for key, value in result.items() if key != 'reports'}
print('FINAL RESULT')
print(json.dumps(summary, ensure_ascii=False, indent=2))